### 🔑 4-3. 연습문제 정답

#### 연습문제 1 정답

In [ ]:
import optuna

def objective_poly(trial):
    x = trial.suggest_float('x', -1, 4)
    return x**3 - 6*x**2 + 9*x + 1

study_poly = optuna.create_study(direction='minimize')
study_poly.optimize(objective_poly, n_trials=50)

print("Best params:", study_poly.best_params)
print("Best value:", study_poly.best_value)

# 이론적 최솟값은 x=3일 때 f(3)=1 입니다. Optuna가 3에 가까운 값을 찾아냅니다.

#### 연습문제 2 정답

In [ ]:
import optuna
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

X, y = fetch_california_housing(return_X_y=True)

def objective_lgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    num_leaves = trial.suggest_int('num_leaves', 20, 300)

    model = LGBMRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        random_state=42,
        n_jobs=-1 # 학습 속도 향상을 위해 추가
    )

    score = cross_val_score(model, X, y, cv=3, scoring='neg_root_mean_squared_error')
    neg_rmse = score.mean()

    return neg_rmse

study_lgbm = optuna.create_study(direction='maximize')
# Pruning (가지치기) 콜백을 추가하여 유망하지 않은 trial을 조기 종료할 수 있습니다.
# study_lgbm.optimize(objective_lgbm, n_trials=50, callbacks=[optuna.integration.LightGBMPruningCallback(trial, 'neg_root_mean_squared_error')])
study_lgbm.optimize(objective_lgbm, n_trials=50)

print("Best Negative RMSE:", study_lgbm.best_value)
print("Best RMSE:", -study_lgbm.best_value)
print("Best Hyperparameters:", study_lgbm.best_params)

#### 연습문제 3 정답

In [ ]:
from optuna.visualization import plot_contour

# study_lgbm 객체가 정의되어 있다고 가정합니다.
plot_contour(study_lgbm, params=['learning_rate', 'num_leaves'])